[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter9/graph-query.ipynb)

# Chapter 9: Hybrid Vector and Graph Querying

This notebook demonstrates the advantage of combining graph-based retrieval with vector search.

**What you'll learn:**
- Populate ChromaDB with graph-extracted chunks for vector search
- Generate Cypher queries from natural language using an LLM
- Compare vector-only vs hybrid (graph + vector) retrieval approaches
- See how graph metadata improves answer quality for entity-centric questions

**Prerequisites:** `pip install langchain langchain-openai langchain-chroma neo4j` and set `OPENAI_API_KEY`. Run `create-graph.ipynb` first.

## Setup and Configuration

Load dependencies and configure connection details for Neo4j and ChromaDB so the hybrid query system can access both stores.

In [1]:
import os
from typing import List, Dict, Any
from neo4j import GraphDatabase
from pydantic import BaseModel
from tqdm import tqdm

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# Import from the unified graph query generator module
from graph_query_generator import GraphQueryGenerator, GRAPH_SCHEMA, CypherQuery, CypherQueryError

# Configuration
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "hands-on-rag-book"
CHROMA_DB_PATH = "./data/chroma_db"
COLLECTION_NAME = "movie_chunks"

## Initialize Vector Store (ChromaDB)

Create a persistent ChromaDB collection and, if it is empty, bulk-ingest every movie chunk from Neo4j so we have a vector index ready for similarity search.

In [2]:
def populate_chromadb(vector_store):
    """Populate ChromaDB with ALL chunks from Neo4j."""
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    total = 0
    batch_size = 1000
    offset = 0
    
    # Create progress bar
    pbar = tqdm(desc="Ingesting chunks", unit="chunks", unit_scale=True)
    
    try:
        while True:
            with driver.session() as session:
                result = session.run(f"""
                    MATCH (ch:Chunk)-[:BELONGS_TO]->(m:Movie)
                    RETURN ch.chunk_id as chunk_id, ch.text as text,
                           ch.chunk_index as chunk_index, m.title as movie_title,
                           m.imdb_id as movie_id, m.start_year as year
                    ORDER BY m.imdb_id, ch.chunk_index
                    SKIP {offset} LIMIT {batch_size}
                """)
                records = list(result)
                
                if not records:
                    break
                
                docs = [Document(
                    page_content=r['text'],
                    metadata={
                        'chunk_id': r['chunk_id'], 
                        'chunk_index': r['chunk_index'],
                        'movie_title': r['movie_title'], 
                        'movie_id': r['movie_id'],
                        'year': r['year']
                    }
                ) for r in records]
                
                vector_store.add_documents(docs)
                total += len(docs)
                offset += len(docs)
                pbar.update(len(docs))
                
                if len(records) < batch_size:
                    break
        
        pbar.close()
        print(f"Total: {total:,} chunks ingested into ChromaDB")
    finally:
        driver.close()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DB_PATH
)

# Check if we need to populate
collection_count = vector_store._collection.count()

if collection_count == 0:
    print("📥 Populating ChromaDB from Neo4j...")
    populate_chromadb(vector_store)
    collection_count = vector_store._collection.count()
    print(f"ChromaDB now contains: {collection_count:,} chunks")
else:
    print(f"ChromaDB already populated with {collection_count:,} chunks")

📥 Populating ChromaDB from Neo4j...


Ingesting chunks: 250kchunks [28:37, 146chunks/s] 

Total: 250,393 chunks ingested into ChromaDB
ChromaDB now contains: 250,393 chunks


## Query System

Combines graph and vector retrieval with comparison capability

In [3]:
class HybridQuerySystem:
    def __init__(self, graph_gen: GraphQueryGenerator, vector_store: Chroma):
        self.graph_gen = graph_gen
        self.vector_store = vector_store
        self.llm = ChatOpenAI(model="gpt-4.1", temperature=0)
    
    def query_graph(self, question: str, limit: int = 10, max_chunks: int = 10) -> Dict[str, Any]:
        """Execute graph query and return metadata + chunks. Uses self-correction from GraphQueryGenerator.

        Args:
            question: Natural language question
            limit: Limit passed to Cypher query generation (chunks per record)
            max_chunks: Maximum total chunks to return across all records
        """
        # Use the unified module's query_with_retry method which includes self-correction
        cypher_result, results = self.graph_gen.query_with_retry(question, limit=limit, verbose=True)

        # Extract metadata and chunks
        metadata = []
        all_chunks = []

        for record in results:
            # Get movie title or other metadata
            movie_title = record.get('movie_title') or record.get('m.title', 'Unknown')
            metadata.append(movie_title)

            # Get chunks (but respect max_chunks limit)
            if len(all_chunks) >= max_chunks:
                break

            chunks = record.get('chunks', [])
            if isinstance(chunks, list):
                for chunk in chunks:
                    if chunk and len(all_chunks) < max_chunks:
                        all_chunks.append(chunk)
                    if len(all_chunks) >= max_chunks:
                        break
            elif chunks:  # Single chunk
                if len(all_chunks) < max_chunks:
                    all_chunks.append(chunks)

        print(f"   → Limited to {len(all_chunks)} total chunks (max_chunks={max_chunks})")

        return {
            'metadata': list(set(metadata)),  # Unique movie titles
            'chunks': all_chunks,
            'raw_results': results
        }
    
    def query_vector(self, question: str, k: int = 10) -> List[Document]:
        """Execute vector similarity search."""
        docs = self.vector_store.similarity_search(question, k=k)
        print(f"🔍 Vector Search:")
        print(f"   → Retrieved {len(docs)} chunks from ChromaDB")
        return docs
    
    def generate_answer(self, question: str, context: str) -> str:
        """Generate answer using LLM with provided context."""
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a movie expert. Answer questions based on the provided context.
            If the context includes movie titles or metadata, use that information in your answer.
            Your response should be based on information provided in the context. Do not use your own internal knowledge.
            If you cannot answer based on the context, say 'I cannot answer this question'."""),
            ("user", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
        ])
        
        response = (prompt | self.llm).invoke({"context": context, "question": question})
        return response.content
    
    def compare_approaches(self, question: str, graph_limit: int = 10, vector_k: int = 10, max_graph_chunks: int = 10) -> Dict[str, Any]:
        """Compare vector-only vs vector+graph hybrid approach.
        
        Args:
            question: Natural language question
            graph_limit: Limit passed to Cypher query (chunks per record)
            vector_k: Number of chunks to retrieve from vector store
            max_graph_chunks: Maximum total chunks from graph across all records
        """
        
        print("\n" + "="*80)
        print(f"QUESTION: {question}")
        print("="*80)
        
        # Get graph results
        print("\n🔍 Step 1: Query graph...")
        graph_data = self.query_graph(question, limit=graph_limit, max_chunks=max_graph_chunks)
        
        # Get vector results
        print(f"\n🔍 Step 2: Query vector store...")
        vector_docs = self.query_vector(question, k=vector_k)
        vector_chunks = [doc.page_content for doc in vector_docs]
        
        print(f"\n📝 Step 3: Generate answers...")
        print(f"   - Graph found movies: {graph_data['metadata']}")
        print(f"   - Graph chunks: {len(graph_data['chunks'])}")
        print(f"   - Vector chunks: {len(vector_chunks)}")
        
        # Prepare contexts
        vector_context = "\n\n".join(vector_chunks)
        
        # Combine graph and vector chunks for hybrid approach
        hybrid_context_parts = [f"Movies found: {', '.join(graph_data['metadata'])}"]
        if graph_data['chunks']:
            hybrid_context_parts.extend(graph_data['chunks'])
        if vector_chunks:
            hybrid_context_parts.extend(vector_chunks)
        
        hybrid_context = "\n\n".join(hybrid_context_parts)
        
        # Generate answers
        vector_answer = self.generate_answer(question, vector_context)
        hybrid_answer = self.generate_answer(question, hybrid_context)
        
        return {
            'question': question,
            'graph_metadata': graph_data['metadata'],
            'graph_chunks': len(graph_data['chunks']),
            'vector_chunks': len(vector_chunks),
            'vector_answer': vector_answer,
            'hybrid_answer': hybrid_answer
        }

graph_gen = GraphQueryGenerator(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
hybrid_system = HybridQuerySystem(graph_gen, vector_store)

## Example 1: Query that requires Graph data

This actor-filmography question shows where vector search alone falls short, because the answer depends on structured relationships (person acted-in movie) that the graph captures directly.

In [4]:
result1 = hybrid_system.compare_approaches(
    "What movies did Pierce Brosnan act in?",
    max_graph_chunks=25,
)

print("\n" + "="*80)
print("COMPARISON RESULTS")
print("="*80)

print(f"\n📊 Graph Retrieved:")
print(f"  - Movies found: {result1['graph_metadata']}")

print(f"\n📊 Vector Retrieved:")
print(f"  - Chunks: {result1['vector_chunks']}")

print(f"\n{'VECTOR-ONLY ANSWER':-^80}")
print(result1['vector_answer'])

print(f"\n{'HYBRID (GRAPH) ANSWER':-^80}")
print(result1['hybrid_answer'])


QUESTION: What movies did Pierce Brosnan act in?

🔍 Step 1: Query graph...
Generated Cypher Query:
   MATCH (p:Person)-[a:ACTED_IN]->(m:Movie)
WHERE toLower(p.name) = 'pierce brosnan'
WITH DISTINCT m.title AS movie_title, m
OPTIONAL MATCH (ch:Chunk)-[:BELONGS_TO]->(m)
WITH movie_title, collect(ch.text)[0..10] AS chunks
RETURN movie_title, chunks
   → Retrieved 4 results from graph
   → Limited to 25 total chunks (max_chunks=25)

🔍 Step 2: Query vector store...
🔍 Vector Search:
   → Retrieved 10 chunks from ChromaDB

📝 Step 3: Generate answers...
   - Graph found movies: ['The World Is Not Enough', 'GoldenEye', 'A Long Way Down', 'Tomorrow Never Dies']
   - Graph chunks: 25
   - Vector chunks: 10

COMPARISON RESULTS

📊 Graph Retrieved:
  - Movies found: ['The World Is Not Enough', 'GoldenEye', 'A Long Way Down', 'Tomorrow Never Dies']

📊 Vector Retrieved:
  - Chunks: 10

-------------------------------VECTOR-ONLY ANSWER-------------------------------
Based on the context provided, Pier

## Example 2: Complex Query about GoldenEye

A multi-hop question that asks the graph to find all characters in a film and then filter those who co-occur with Bond in the same chunk, demonstrating richer relational reasoning.

In [5]:
result2 = hybrid_system.compare_approaches(
    "What are all the characters in GoldenEye? Which of them interacted with Bond",
    max_graph_chunks=25,
)

print("\n" + "="*80)
print("COMPARISON RESULTS")
print("="*80)

print(f"\n📊 Graph Retrieved:")
print(f"  - Movies found: {result2['graph_metadata']}")

print(f"\n📊 Vector Retrieved:")
print(f"  - Chunks: {result2['vector_chunks']}")

print(f"\n{'VECTOR-ONLY ANSWER':-^80}")
print(result2['vector_answer'])

print(f"\n{'HYBRID (GRAPH) ANSWER':-^80}")
print(result2['hybrid_answer'])


QUESTION: What are all the characters in GoldenEye? Which of them interacted with Bond

🔍 Step 1: Query graph...
Generated Cypher Query:
   // Step 1: Find all Character nodes that appear in GoldenEye
MATCH (m:Movie)
WHERE toLower(m.title) CONTAINS 'goldeneye'
WITH m
MATCH (c:Character)-[:APPEARS_IN]->(m)
WITH m, collect(DISTINCT c.name) AS all_characters, m AS movie

// Step 2: Find all script chunks in GoldenEye where Bond interacts with another character
// We'll use fuzzy matching for 'bond' to find Bond's character node(s)
MATCH (movie)
MATCH (ch:Chunk)-[:BELONGS_TO]->(movie)
// Chunks mentioning Bond
MATCH (ch)-[:MENTIONS]->(bondChar:Character)
WHERE toLower(bondChar.name) CONTAINS 'bond'
// Chunks also mentioning another character (not Bond)
MATCH (ch)-[:MENTIONS]->(otherChar:Character)
WHERE otherChar <> bondChar
WITH movie, all_characters, collect(DISTINCT otherChar.name) AS interacted_characters, collect(DISTINCT ch.text)[0..10] AS chunks
RETURN movie.title AS movie_title, a

## Cleanup

Close the Neo4j driver connection to release resources now that all queries are complete.

In [6]:
graph_gen.close()